# Fine-tuning: Unified Training & Evaluation Pipeline

Цель ноутбука — реализовать единый воспроизводимый pipeline для downstream-экспериментов:
- Scratch
- SSL + Fine-Tuning

Pipeline должен:
- работать по субъектам
- поддерживать calibration levels p
- включать train/val split
- использовать early stopping
- считать итоговые метрики
- сохранять результаты и предсказания

## 1. Импорты

In [13]:
import os
import json
import math
import random
import gc
from pathlib import Path
from copy import deepcopy

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)

## 2. Global config

In [14]:
CONFIG = {
    "seed": 42,
    
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "pin_memory": torch.cuda.is_available(),

    "data_root": Path("/kaggle/input/datasets/taisiyaglazova"),
    "encoder_checkpoint": Path("/kaggle/input/datasets/taisiyaglazova/ssl-full-encoder-best/encoder_best.pt"),
    "results_root": Path("/kaggle/working/stage5_block2_results"),

    "batch_size": 64,
    "num_workers": 2,

    "max_epochs": 100,
    "patience": 10,
    "min_delta": 0.0,

    "lr": 1e-4,
    "weight_decay": 1e-4,

    "fallback_p_for_zero": 10,
    "val_ratio": 0.2,

    "p_list": [0, 10, 20, 40, 60, 100],
    "scenarios": ["scratch", "ssl_ft"],

    "save_predictions": True,
    "save_history": True,
}

In [15]:
# Выходные папки
(CONFIG["results_root"] / "history").mkdir(parents=True, exist_ok=True)
(CONFIG["results_root"] / "predictions").mkdir(parents=True, exist_ok=True)
(CONFIG["results_root"] / "checkpoints").mkdir(parents=True, exist_ok=True)
(CONFIG["results_root"] / "logs").mkdir(parents=True, exist_ok=True)

## 3. Воспроизводимость

In [16]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

In [17]:
set_seed(CONFIG["seed"])
print("Device:", CONFIG["device"])

Device: cuda


## 4. Пути

In [18]:
# Пути к датасетам
DATASETS = {
    "bigp3_train": CONFIG["data_root"] / "bigp3bci-downstream-train",
    "bigp3_benchmark": CONFIG["data_root"] / "bigp3bci-downstream-benchmark",
    "bcicomp3": CONFIG["data_root"] / "bcicompiii-ds2",  
}

In [19]:
GROUPS = ["train", "benchmark"]

In [20]:
# Универсальные функции путей
def get_epochs_path(subject_id: str, group: str) -> Path:
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        path = DATASETS["bigp3_train"] / "train" / f"{subject_id}.npz"
    elif group == "benchmark":
        path = DATASETS["bigp3_benchmark"]/ "benchmark" / f"{subject_id}.npz"

    return path


def get_split_path(subject_id: str, group: str) -> Path:
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        path = DATASETS["bigp3_train"] / "splits" / "train" / f"{subject_id}.json"
    elif group == "benchmark":
        path = DATASETS["bigp3_benchmark"] / "splits" / "benchmark" / f"{subject_id}.json"

    return path


def get_stats_path(subject_id: str, p: int, group: str) -> Path:
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        path = DATASETS["bigp3_train"] / "stats" / "train" / f"{subject_id}_p{p}.npz"
    elif group == "benchmark":
        path = DATASETS["bigp3_benchmark"] / "stats" / "benchmark" / f"{subject_id}_p{p}.npz"

    return path

In [21]:
# Автореестр субъектов
def list_subject_ids(group: str):
    assert group in GROUPS, f"Unknown group: {group}"

    if group == "train":
        data_dir = DATASETS["bigp3_train"] / "train"
    elif group == "benchmark":
        data_dir = DATASETS["bigp3_benchmark"]/ "benchmark"

    subject_ids = sorted([p.stem for p in data_dir.glob("*.npz")])
    return subject_ids

In [22]:
# Сборка реестра
SUBJECT_REGISTRY = {
    "train": list_subject_ids("train"),
    "benchmark": list_subject_ids("benchmark"),
}

print("Train subjects:", len(SUBJECT_REGISTRY["train"]))
print("Benchmark subjects:", len(SUBJECT_REGISTRY["benchmark"]))
print("Example train:", SUBJECT_REGISTRY["train"][:5])
print("Example benchmark:", SUBJECT_REGISTRY["benchmark"][:5])


Train subjects: 93
Benchmark subjects: 65
Example train: ['subj_000', 'subj_001', 'subj_002', 'subj_003', 'subj_004']
Example benchmark: ['subj_051', 'subj_052', 'subj_053', 'subj_054', 'subj_055']


In [23]:
# Валидация существования файлов
def check_subject_files(subject_id: str, group: str, p_list=(10, 20, 40, 60, 100)):
    report = {
        "epochs": get_epochs_path(subject_id, group).exists(),
        "split": get_split_path(subject_id, group).exists(),
        "stats": {p: get_stats_path(subject_id, p, group).exists() for p in p_list},
    }
    return report

example_subj = SUBJECT_REGISTRY["train"][80]
check_subject_files(example_subj, "train")

{'epochs': True,
 'split': True,
 'stats': {10: True, 20: True, 40: True, 60: True, 100: True}}

## 5. Core data I/O

In [24]:
# Загрузка эпох
def load_subject_epochs(subject_id: str, group: str):
    path = get_epochs_path(subject_id, group)
    if not path.exists():
        raise FileNotFoundError(f"Epochs file not found: {path}")

    data = np.load(path, allow_pickle=True)

    if "X" not in data or "y" not in data:
        raise KeyError(f"{path} must contain keys 'X' and 'y'. Found: {list(data.keys())}")

    X = data["X"]
    y = data["y"]

    return X, y

In [25]:
# Загрузка split
def load_subject_split(subject_id: str, group: str):
    path = get_split_path(subject_id, group)
    if not path.exists():
        raise FileNotFoundError(f"Split file not found: {path}")

    with open(path, "r", encoding="utf-8") as f:
        split = json.load(f)

    return split

In [26]:
# Загрузка stats
def load_subject_stats(subject_id: str, p: int, group: str):
    path = get_stats_path(subject_id, p, group)
    if not path.exists():
        raise FileNotFoundError(f"Stats file not found: {path}")

    data = np.load(path, allow_pickle=True)

    if "mean" not in data or "std" not in data:
        raise KeyError(f"{path} must contain keys 'mean' and 'std'. Found: {list(data.keys())}")

    mean = data["mean"]
    std = data["std"]

    return mean, std

In [27]:
# Объединённая загрузка bundle
def load_subject_bundle(subject_id: str, p: int, group: str):
    X, y = load_subject_epochs(subject_id, group)
    split = load_subject_split(subject_id, group)

    mean, std = (None, None)
    if p > 0:
        mean, std = load_subject_stats(subject_id, p, group)

    bundle = {
        "subject_id": subject_id,
        "group": group,
        "p": p,
        "X": X,
        "y": y,
        "split": split,
        "mean": mean,
        "std": std,
    }
    return bundle

## Split внутри Calib_p

In [28]:
# Функции для извлечения индексов
def get_test_indices(split: dict) -> np.ndarray:
    """
    Возвращает индексы test_rest в глобальной индексации subject-level X/y.
    """
    if "test_rest_idx" not in split:
        raise KeyError("split must contain 'test_rest_idx'")
    return np.asarray(split["test_rest_idx"], dtype=np.int64)


def get_calib_pool_indices(split: dict) -> np.ndarray:
    """
    Возвращает индексы calib_pool в глобальной индексации subject-level X/y.
    """
    if "calib_pool_idx" not in split:
        raise KeyError("split must contain 'calib_pool_idx'")
    return np.asarray(split["calib_pool_idx"], dtype=np.int64)


def get_calib_indices(split: dict, p: int) -> np.ndarray:
    """
    Возвращает индексы calib_p в глобальной индексации subject-level X/y.

    Для p=0 возвращает пустой массив.
    """
    if p == 0:
        return np.asarray([], dtype=np.int64)

    if "calib_idx" not in split:
        raise KeyError("split must contain 'calib_idx'")

    calib_idx_dict = split["calib_idx"]

    if not isinstance(calib_idx_dict, dict):
        raise TypeError(f"split['calib_idx'] must be dict, got {type(calib_idx_dict)}")

    if str(p) not in calib_idx_dict:
        raise KeyError(
            f"p={p} not found in split['calib_idx']; available keys: {list(calib_idx_dict.keys())}"
        )

    return np.asarray(calib_idx_dict[str(p)], dtype=np.int64)


def make_train_val_split(
    calib_idx: np.ndarray,
    y: np.ndarray,
    val_ratio: float = 0.2,
    seed: int = 42,
    stratify: bool = True,
):
    """
    Делит calib_idx на train_idx и val_idx.

    Все индексы остаются глобальными относительно X/y конкретного subject.
    """
    calib_idx = np.asarray(calib_idx, dtype=np.int64)

    if len(calib_idx) == 0:
        return (
            np.asarray([], dtype=np.int64),
            np.asarray([], dtype=np.int64),
        )

    if len(calib_idx) < 2:
        raise ValueError("calib_idx must contain at least 2 samples")

    y_calib = y[calib_idx]
    stratify_labels = y_calib if stratify else None

    try:
        train_idx, val_idx = train_test_split(
            calib_idx,
            test_size=val_ratio,
            random_state=seed,
            shuffle=True,
            stratify=stratify_labels,
        )
    except ValueError as e:
        print(f"[WARN] Stratified split failed: {e}")
        print("[WARN] Falling back to non-stratified split.")
        train_idx, val_idx = train_test_split(
            calib_idx,
            test_size=val_ratio,
            random_state=seed,
            shuffle=True,
            stratify=None,
        )

    return (
        np.asarray(train_idx, dtype=np.int64),
        np.asarray(val_idx, dtype=np.int64),
    )


def prepare_run_indices(
    split: dict,
    y: np.ndarray,
    p: int,
    val_ratio: float = 0.2,
    seed: int = 42,
):
    """
    Готовит все индексы для одного запуска.

    Возвращает словарь:
    - calib_pool_idx
    - calib_idx
    - train_idx
    - val_idx
    - test_idx
    """
    calib_pool_idx = get_calib_pool_indices(split)
    test_idx = get_test_indices(split)
    calib_idx = get_calib_indices(split, p)

    calib_pool_set = set(calib_pool_idx.tolist())
    test_set = set(test_idx.tolist())
    calib_set = set(calib_idx.tolist())

    # test и calib_pool не должны пересекаться
    if len(calib_pool_set & test_set) > 0:
        raise ValueError("calib_pool_idx intersects with test_rest_idx")

    # calib_p должен быть подмножеством calib_pool
    if len(calib_set) > 0 and not calib_set.issubset(calib_pool_set):
        raise ValueError("calib_idx is not a subset of calib_pool_idx")

    if p == 0:
        train_idx = np.asarray([], dtype=np.int64)
        val_idx = np.asarray([], dtype=np.int64)
    else:
        train_idx, val_idx = make_train_val_split(
            calib_idx=calib_idx,
            y=y,
            val_ratio=val_ratio,
            seed=seed,
            stratify=True,
        )

    return {
        "calib_pool_idx": calib_pool_idx,
        "calib_idx": calib_idx,
        "train_idx": train_idx,
        "val_idx": val_idx,
        "test_idx": test_idx,
    }

## Нормализация и нарезка данных по индексам

In [29]:
# Фукции нормализации
def safe_standardize(X: np.ndarray, mean: np.ndarray, std: np.ndarray, eps: float = 1e-8) -> np.ndarray:
    """
    Поканальная z-нормализация массива X формы (N, C, L)
    по mean/std формы (C,).
    """
    X = np.asarray(X, dtype=np.float32)
    mean = np.asarray(mean, dtype=np.float32)
    std = np.asarray(std, dtype=np.float32)

    if X.ndim != 3:
        raise ValueError(f"X must have shape (N, C, L), got {X.shape}")
    if mean.ndim != 1 or std.ndim != 1:
        raise ValueError(f"mean/std must have shape (C,), got mean={mean.shape}, std={std.shape}")
    if X.shape[1] != len(mean) or X.shape[1] != len(std):
        raise ValueError(
            f"Channel mismatch: X has C={X.shape[1]}, mean={len(mean)}, std={len(std)}"
        )

    std_safe = np.maximum(std, eps)
    X_norm = (X - mean[None, :, None]) / std_safe[None, :, None]
    return X_norm.astype(np.float32)


def get_effective_stats(bundle: dict, subject_id: str, group: str, p: int, fallback_p_for_zero: int = 10):
    """
    Возвращает mean/std для данного запуска.
    
    Для p>0 используются stats именно этого p.
    Для p=0 используются fallback-статистики, например p=10.
    """
    if p > 0:
        mean = bundle["mean"]
        std = bundle["std"]
    else:
        mean, std = load_subject_stats(subject_id, fallback_p_for_zero, group)

    if mean is None or std is None:
        raise ValueError(f"Stats are not available for subject={subject_id}, group={group}, p={p}")

    return mean, std

In [30]:
# Функции нарезки по индексам
def slice_by_indices(X: np.ndarray, y: np.ndarray, idx: np.ndarray):
    """
    Возвращает подмножество X/y по глобальным индексам subject-level массива.
    """
    idx = np.asarray(idx, dtype=np.int64)

    if len(idx) == 0:
        X_empty = np.empty((0, X.shape[1], X.shape[2]), dtype=np.float32)
        y_empty = np.empty((0,), dtype=y.dtype)
        return X_empty, y_empty

    return X[idx], y[idx]


def prepare_indexed_arrays(
    bundle: dict,
    indices_dict: dict,
    fallback_p_for_zero: int = 10,
):
    """
    Подготавливает нормализованные массивы:
    - X_train, y_train
    - X_val, y_val
    - X_test, y_test

    bundle должен содержать:
    - subject_id
    - group
    - p
    - X
    - y
    - mean/std (если p>0)
    """
    subject_id = bundle["subject_id"]
    group = bundle["group"]
    p = bundle["p"]

    X = bundle["X"]
    y = bundle["y"]

    mean, std = get_effective_stats(
        bundle=bundle,
        subject_id=subject_id,
        group=group,
        p=p,
        fallback_p_for_zero=fallback_p_for_zero,
    )

    X_norm = safe_standardize(X, mean, std)

    X_train, y_train = slice_by_indices(X_norm, y, indices_dict["train_idx"])
    X_val, y_val = slice_by_indices(X_norm, y, indices_dict["val_idx"])
    X_test, y_test = slice_by_indices(X_norm, y, indices_dict["test_idx"])

    return {
        "X_train": X_train,
        "y_train": y_train,
        "X_val": X_val,
        "y_val": y_val,
        "X_test": X_test,
        "y_test": y_test,
        "mean": mean,
        "std": std,
    }

## 6. Dataset / DataLoader

In [31]:
class EEGDataset(Dataset):
    """
    Простой Dataset для EEG-эпох.
    X: np.ndarray формы (N, C, L)
    y: np.ndarray формы (N,)
    """
    def __init__(self, X: np.ndarray, y: np.ndarray):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y)

        if X.ndim != 3:
            raise ValueError(f"X must have shape (N, C, L), got {X.shape}")
        if y.ndim != 1:
            raise ValueError(f"y must have shape (N,), got {y.shape}")
        if len(X) != len(y):
            raise ValueError(f"Length mismatch: len(X)={len(X)}, len(y)={len(y)}")

        self.X = X
        self.y = y.astype(np.int64)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = torch.from_numpy(self.X[idx])      # float32, shape (C, L)
        y = torch.tensor(self.y[idx], dtype=torch.long)
        return x, y

In [32]:
def make_loader(
    X: np.ndarray,
    y: np.ndarray,
    batch_size: int,
    shuffle: bool,
    num_workers: int = 0,
    pin_memory: bool = True,
):
    """
    Создаёт DataLoader для заданных X/y.
    Если массив пустой, возвращает None.
    """
    if len(y) == 0:
        return None

    dataset = EEGDataset(X, y)

    loader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=num_workers,
        pin_memory=pin_memory,
        drop_last=False,
    )
    return loader

In [33]:
def build_loaders(
    arrays_dict: dict,
    batch_size: int = 64,
    num_workers: int = 0,
    pin_memory: bool = True,
):
    """
    По словарю prepared arrays создаёт:
    - train_loader
    - val_loader
    - test_loader

    Для p=0 train/val будут None.
    """
    train_loader = make_loader(
        X=arrays_dict["X_train"],
        y=arrays_dict["y_train"],
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    val_loader = make_loader(
        X=arrays_dict["X_val"],
        y=arrays_dict["y_val"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    test_loader = make_loader(
        X=arrays_dict["X_test"],
        y=arrays_dict["y_test"],
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
    )

    return {
        "train_loader": train_loader,
        "val_loader": val_loader,
        "test_loader": test_loader,
    }

## 7. Определение модели

In [34]:
class DoubleConv1D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv1d(in_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv1d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class Down1D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.MaxPool1d(kernel_size=2, stride=2),
            DoubleConv1D(in_channels, out_channels)
        )

    def forward(self, x):
        return self.block(x)


class Up1D(nn.Module):
    def __init__(self, in_channels, out_channels, bilinear=True):
        super().__init__()
        self.bilinear = bilinear

        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="linear", align_corners=True)
            mid_channels = in_channels // 2
            self.conv = DoubleConv1D(in_channels, out_channels)
        else:
            self.up = nn.ConvTranspose1d(in_channels, out_channels, kernel_size=2, stride=2)
            self.conv = DoubleConv1D(in_channels, out_channels)

    def forward(self, x1, x2):
        # x1: низ, x2: skip
        x1 = self.up(x1)

        # подгоняем длину, если не совпадает
        diff = x2.size(-1) - x1.size(-1)
        if diff > 0:
            x1 = nn.functional.pad(x1, (diff // 2, diff - diff // 2))
        elif diff < 0:
            x2 = nn.functional.pad(x2, (-diff // 2, -diff - (-diff // 2)))

        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)


class OutConv1D(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.conv = nn.Conv1d(in_channels, out_channels, kernel_size=1)

    def forward(self, x):
        return self.conv(x)


class UNet1D_Light(nn.Module):
    """
    U-Net 1D с 4 уровнями даунсемплинга, вдохновлён Hong et al., 2025.
    Каналы: 32 → 64 → 128 → 256, bottleneck 512.
    """
    def __init__(self, n_channels, n_classes, base_ch=32, bilinear=True):
        super().__init__()
        self.n_channels = n_channels
        self.n_classes = n_classes
        self.bilinear = bilinear

        ch1 = base_ch
        ch2 = base_ch * 2
        ch3 = base_ch * 4
        ch4 = base_ch * 8
        bottleneck_ch = base_ch * 16  # 512 при base_ch=32

        # encoder
        self.inc = DoubleConv1D(n_channels, ch1)
        self.down1 = Down1D(ch1, ch2)
        self.down2 = Down1D(ch2, ch3)
        self.down3 = Down1D(ch3, ch4)
        self.down4 = Down1D(ch4, bottleneck_ch)

        # decoder
        self.up1 = Up1D(bottleneck_ch + ch4, ch4, bilinear)
        self.up2 = Up1D(ch4 + ch3, ch3, bilinear)
        self.up3 = Up1D(ch3 + ch2, ch2, bilinear)
        self.up4 = Up1D(ch2 + ch1, ch1, bilinear)
        self.outc = OutConv1D(ch1, n_classes)

    def encode(self, x):
        x1 = self.inc(x)     # (N, ch1, L)
        x2 = self.down1(x1)  # (N, ch2, L/2)
        x3 = self.down2(x2)  # (N, ch3, L/4)
        x4 = self.down3(x3)  # (N, ch4, L/8)
        x5 = self.down4(x4)  # (N, bottleneck_ch, L/16)
        return x1, x2, x3, x4, x5

    def decode(self, x1, x2, x3, x4, x5):
        x = self.up1(x5, x4)
        x = self.up2(x, x3)
        x = self.up3(x, x2)
        x = self.up4(x, x1)
        logits = self.outc(x)
        return logits

    def forward(self, x):
        x1, x2, x3, x4, x5 = self.encode(x)
        logits = self.decode(x1, x2, x3, x4, x5)
        return logits, x5  # x5 — bottleneck

class UNet1DEncoder(nn.Module):
    """
    Обёртка над encoder-частью обученного U-Net.
    На вход:  x ∈ R^{B×C×L}
    На выход: bottleneck-фичи x5 ∈ R^{B×C_bottleneck×L_reduced}
    """
    def __init__(self, unet_model: nn.Module):
        super().__init__()
        # просто переиспользуем уже обученные блоки
        self.inc = unet_model.inc
        self.down1 = unet_model.down1
        self.down2 = unet_model.down2
        self.down3 = unet_model.down3
        self.down4 = unet_model.down4

    def forward(self, x):
        # это ровно то, что делал encode() в полном U-Net
        x1 = self.inc(x)    # (B, ch1, L)
        x2 = self.down1(x1) # (B, ch2, L/2)
        x3 = self.down2(x2) # (B, ch3, L/4)
        x4 = self.down3(x3) # (B, ch4, L/8)
        x5 = self.down4(x4) # (B, bottleneck_ch, L/16)
        return x5

class ERPHead(nn.Module):
    """
    Минимальная классификационная голова для P300.
    Вход:  (B, F, T)
    Выход: (B, 2) logits
    """
    def __init__(self, in_features=512, num_classes=2):
        super().__init__()
        self.fc = nn.Linear(in_features, num_classes)

    def forward(self, z):
        # z: (B, F, T)
        z = z.mean(dim=-1)   # global average pooling → (B, F)
        logits = self.fc(z)  # (B, 2)
        return logits
    
class P300Model(nn.Module):
    """
    Полная downstream модель:
    x → encoder → head → logits
    """
    def __init__(self, encoder, head):
        super().__init__()
        self.encoder = encoder
        self.head = head

    def forward(self, x):
        z = self.encoder(x)
        logits = self.head(z)
        return logits

In [35]:
# Загрузка весов
def load_encoder_checkpoint_into_model_encoder(model_encoder: nn.Module, encoder_checkpoint: str, device: str = "cpu"):
    """
    Загружает encoder_best.pt в encoder downstream-модели.

    Ожидаемый формат checkpoint:
    {
        'inc': state_dict(...),
        'down1': state_dict(...),
        'down2': state_dict(...),
        'down3': state_dict(...),
        'down4': state_dict(...),
    }
    """
    ckpt = torch.load(encoder_checkpoint, map_location=device)

    expected_keys = ["inc", "down1", "down2", "down3", "down4"]
    missing = [k for k in expected_keys if k not in ckpt]
    if len(missing) > 0:
        raise KeyError(f"Encoder checkpoint is missing keys: {missing}. Found keys: {list(ckpt.keys())}")

    model_encoder.inc.load_state_dict(ckpt["inc"], strict=True)
    model_encoder.down1.load_state_dict(ckpt["down1"], strict=True)
    model_encoder.down2.load_state_dict(ckpt["down2"], strict=True)
    model_encoder.down3.load_state_dict(ckpt["down3"], strict=True)
    model_encoder.down4.load_state_dict(ckpt["down4"], strict=True)

    return model_encoder

In [36]:
def build_model(
    scenario: str,
    device: str,
    encoder_checkpoint: str = None,
):
    """
    Собирает downstream-модель для сценариев:
    - scratch
    - ssl_ft

    scratch:
        encoder случайный

    ssl_ft:
        encoder той же архитектуры + загруженные SSL-веса

    head всегда создаётся заново
    """
    valid_scenarios = {"scratch", "ssl_ft"}
    if scenario not in valid_scenarios:
        raise ValueError(f"Unknown scenario: {scenario}. Expected one of {valid_scenarios}")

    # создаём базовый U-Net только как контейнер encoder-блоков
    unet = UNet1D_Light(
        n_channels=14,
        n_classes=14,
        base_ch=32,
        bilinear=True,
    )
    encoder = UNet1DEncoder(unet)

    if scenario == "ssl_ft":
        if encoder_checkpoint is None:
            raise ValueError("encoder_checkpoint must be provided for scenario='ssl_ft'")
        encoder = load_encoder_checkpoint_into_model_encoder(
            model_encoder=encoder,
            encoder_checkpoint=encoder_checkpoint,
            device=device,
        )

    head = ERPHead(in_features=512, num_classes=2)
    model = P300Model(encoder=encoder, head=head)
    model = model.to(device)

    return model

In [59]:
# Счётчик параметров
def count_all_parameters(model: nn.Module):
    return sum(p.numel() for p in model.parameters())


def count_trainable_parameters(model: nn.Module):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

## 8. Training / evaluation 

In [37]:
def train_one_epoch(model, loader, optimizer, criterion, device: str):
    """
    Одна эпоха обучения.
    Возвращает средний loss по эпохе.
    """
    if loader is None:
        raise ValueError("train loader is None")

    model.train()

    running_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(xb)
        loss = criterion(logits, yb)

        loss.backward()
        optimizer.step()

        batch_size = xb.size(0)
        running_loss += loss.item() * batch_size
        n_samples += batch_size

    epoch_loss = running_loss / max(n_samples, 1)
    return epoch_loss


@torch.no_grad()
def validate_one_epoch(model, loader, criterion, device: str):
    """
    Одна эпоха валидации.
    Возвращает средний loss по эпохе.
    """
    if loader is None:
        raise ValueError("val loader is None")

    model.eval()

    running_loss = 0.0
    n_samples = 0

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        logits = model(xb)
        loss = criterion(logits, yb)

        batch_size = xb.size(0)
        running_loss += loss.item() * batch_size
        n_samples += batch_size

    epoch_loss = running_loss / max(n_samples, 1)
    return epoch_loss

In [38]:
# training history
def init_history():
    return {
        "epoch": [],
        "train_loss": [],
        "val_loss": [],
    }


def append_history(history: dict, epoch: int, train_loss: float, val_loss: float):
    history["epoch"].append(epoch)
    history["train_loss"].append(float(train_loss))
    history["val_loss"].append(float(val_loss))
    return history

In [39]:
# цикл обучения
def fit_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    device: str,
    max_epochs: int = 100,
    patience: int = 10,
    min_delta: float = 0.0,
    verbose: bool = True,
):
    """
    Train/val цикл с early stopping по val_loss.

    Возвращает словарь с:
    - history
    - best_epoch
    - best_val_loss
    - best_state_dict
    - stopped_epoch
    """
    if train_loader is None:
        raise ValueError("train_loader is None")
    if val_loader is None:
        raise ValueError("val_loader is None")

    history = init_history()
    early_stopper = EarlyStopping(
        patience=patience,
        min_delta=min_delta,
        mode="min",
    )

    stopped_epoch = max_epochs

    for epoch in range(1, max_epochs + 1):
        train_loss = train_one_epoch(
            model=model,
            loader=train_loader,
            optimizer=optimizer,
            criterion=criterion,
            device=device,
        )

        val_loss = validate_one_epoch(
            model=model,
            loader=val_loader,
            criterion=criterion,
            device=device,
        )

        append_history(history, epoch, train_loss, val_loss)

        early_stopper.step(val_loss, model, epoch)

        if verbose:
            msg = (
                f"epoch {epoch:03d} | "
                f"train_loss={train_loss:.6f} | "
                f"val_loss={val_loss:.6f} | "
                f"best_val={early_stopper.best_value:.6f} @ epoch {early_stopper.best_epoch}"
            )
            print(msg)

        if early_stopper.should_stop:
            stopped_epoch = epoch
            if verbose:
                print(f"Early stopping triggered at epoch {epoch}.")
            break

    result = {
        "history": history,
        "best_epoch": early_stopper.best_epoch,
        "best_val_loss": early_stopper.best_value,
        "best_state_dict": early_stopper.best_state_dict,
        "stopped_epoch": stopped_epoch,
    }
    return result

In [40]:
# Optimizer и Loss
def build_optimizer(model, lr: float, weight_decay: float):
    return torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr,
        weight_decay=weight_decay,
    )


def build_criterion():
    return nn.CrossEntropyLoss()

In [41]:
# Загрузка best state
def load_best_model_state(model: nn.Module, fit_result: dict):
    """
    Загружает best_state_dict обратно в модель.
    """
    best_state_dict = fit_result.get("best_state_dict", None)
    if best_state_dict is None:
        raise ValueError("fit_result does not contain 'best_state_dict'")
    model.load_state_dict(best_state_dict)
    return model

## 10. Early stopping / checkpoint utilities

In [42]:
class EarlyStopping:
    """
    Early stopping по val_loss.

    mode='min' означает, что метрика должна уменьшаться.
    """
    def __init__(self, patience: int = 10, min_delta: float = 0.0, mode: str = "min"):
        if mode not in {"min", "max"}:
            raise ValueError("mode must be 'min' or 'max'")

        self.patience = patience
        self.min_delta = float(min_delta)
        self.mode = mode

        self.best_value = None
        self.best_epoch = None
        self.best_state_dict = None
        self.counter = 0
        self.should_stop = False

    def _is_improvement(self, value: float) -> bool:
        if self.best_value is None:
            return True

        if self.mode == "min":
            return value < (self.best_value - self.min_delta)
        else:
            return value > (self.best_value + self.min_delta)

    def step(self, value: float, model: nn.Module, epoch: int):
        """
        Обновляет состояние early stopping после очередной эпохи.
        """
        if self._is_improvement(value):
            self.best_value = float(value)
            self.best_epoch = int(epoch)
            self.best_state_dict = deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            if self.counter >= self.patience:
                self.should_stop = True

## Test prediction и метрики

In [43]:
# prediction на loader
@torch.no_grad()
def predict_on_loader(model, loader, device: str):
    """
    Прогоняет модель по loader и возвращает:
    - y_true
    - prob_score   : probability of positive class
    - logit_score  : raw logit of positive class
    - y_pred       : threshold=0.5 on prob_score
    """
    if loader is None:
        raise ValueError("loader is None")

    model.eval()

    all_y = []
    all_prob = []
    all_logit = []
    all_pred = []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)

        logits = model(xb)                    # (B, 2)
        probs = torch.softmax(logits, dim=1)  # (B, 2)

        prob_score = probs[:, 1]
        logit_score = logits[:, 1]
        pred = (prob_score >= 0.5).long()

        all_y.append(yb.numpy())
        all_prob.append(prob_score.cpu().numpy())
        all_logit.append(logit_score.cpu().numpy())
        all_pred.append(pred.cpu().numpy())

    y_true = np.concatenate(all_y) if len(all_y) > 0 else np.array([], dtype=np.int64)
    prob_score = np.concatenate(all_prob) if len(all_prob) > 0 else np.array([], dtype=np.float32)
    logit_score = np.concatenate(all_logit) if len(all_logit) > 0 else np.array([], dtype=np.float32)
    y_pred = np.concatenate(all_pred) if len(all_pred) > 0 else np.array([], dtype=np.int64)

    return {
        "y_true": y_true,
        "prob_score": prob_score,
        "logit_score": logit_score,
        "y_pred": y_pred,
    }

In [44]:
## FDR
def compute_fisher_fdr(y_true: np.ndarray, score: np.ndarray):
    """
    Fisher's Discriminant Ratio:
        FDR = (mu_pos - mu_neg)^2 / (var_pos + var_neg)

    score должен быть непрерывным classifier score.
    """
    y_true = np.asarray(y_true).astype(int)
    score = np.asarray(score).astype(float)

    pos_scores = score[y_true == 1]
    neg_scores = score[y_true == 0]

    if len(pos_scores) == 0 or len(neg_scores) == 0:
        return np.nan

    mu_pos = np.mean(pos_scores)
    mu_neg = np.mean(neg_scores)

    var_pos = np.var(pos_scores)
    var_neg = np.var(neg_scores)

    denom = var_pos + var_neg
    if denom <= 0:
        return np.nan

    return float((mu_pos - mu_neg) ** 2 / denom)

In [45]:
# accuracy, f1, precision, recall, fdr
def compute_metrics(
    y_true: np.ndarray,
    prob_score: np.ndarray,
    logit_score: np.ndarray,
    y_pred: np.ndarray,
):
    """
    Считает метрики в соответствии с логикой статьи:
    - AUC: по probability score
    - Accuracy: по binary predictions
    - F1: по binary predictions
    - Precision/Recall: по binary predictions
    - FDR: Fisher's Discriminant Ratio по classifier score
    """
    y_true = np.asarray(y_true).astype(int)
    prob_score = np.asarray(prob_score).astype(float)
    logit_score = np.asarray(logit_score).astype(float)
    y_pred = np.asarray(y_pred).astype(int)

    metrics = {}

    if len(np.unique(y_true)) < 2:
        metrics["auc"] = np.nan
    else:
        metrics["auc"] = float(roc_auc_score(y_true, prob_score))

    metrics["accuracy"] = float(accuracy_score(y_true, y_pred))
    metrics["f1"] = float(f1_score(y_true, y_pred, zero_division=0))
    metrics["precision"] = float(precision_score(y_true, y_pred, zero_division=0))
    metrics["recall"] = float(recall_score(y_true, y_pred, zero_division=0))
    metrics["fdr"] = compute_fisher_fdr(y_true, logit_score)

    return metrics

In [46]:
# Объединяющая функия test evaluation
def evaluate_on_test(model, test_loader, device: str):
    pred_dict = predict_on_loader(
        model=model,
        loader=test_loader,
        device=device,
    )

    metrics = compute_metrics(
        y_true=pred_dict["y_true"],
        prob_score=pred_dict["prob_score"],
        logit_score=pred_dict["logit_score"],
        y_pred=pred_dict["y_pred"],
    )

    return {
        "predictions": pred_dict,
        "metrics": metrics,
    }

## 11. Main run API

In [47]:
def extract_split_stats(indices_dict: dict, y: np.ndarray):
    def _count(idx):
        idx = np.asarray(idx, dtype=np.int64)
        n = len(idx)
        if n == 0:
            return {"n": 0, "n_pos": 0, "n_neg": 0}
        n_pos = int(y[idx].sum())
        n_neg = int(n - n_pos)
        return {"n": n, "n_pos": n_pos, "n_neg": n_neg}

    train_stats = _count(indices_dict["train_idx"])
    val_stats = _count(indices_dict["val_idx"])
    test_stats = _count(indices_dict["test_idx"])
    calib_stats = _count(indices_dict["calib_idx"])

    return {
        "n_calib": calib_stats["n"],
        "n_val": val_stats["n"],
        "n_test": test_stats["n"],
        "n_pos_calib": calib_stats["n_pos"],
        "n_pos_val": val_stats["n_pos"],
        "n_pos_test": test_stats["n_pos"],
    }

- p =0 → no training, fallback normalization with p=10

- early stopping by val_loss

- metrics include ROC-AUC, Accuracy, Binary F1, Fisher FDR

- same train/val split for scratch and ssl_ft

In [48]:
def run_one(
    subject_id: str,
    p: int,
    scenario: str,
    seed: int,
    group: str,
    encoder_checkpoint: str = None,
):
    """
    Один полный запуск pipeline для одного subject / p / scenario.

    Возвращает словарь:
    - result_row   : итоговая строка результатов
    - history_df   : history по эпохам (или пустой df для p=0)
    - predictions  : y_true / prob_score / logit_score / y_pred
    """
    valid_scenarios = {"scratch", "ssl_ft"}
    if scenario not in valid_scenarios:
        raise ValueError(f"Unknown scenario: {scenario}. Expected one of {valid_scenarios}")

    if scenario == "ssl_ft" and encoder_checkpoint is None:
        raise ValueError("encoder_checkpoint must be provided for scenario='ssl_ft'")

    # reproducibility
    set_seed(seed)

    device = CONFIG["device"]

    # 1) load subject bundle
    bundle = load_subject_bundle(subject_id, p=p, group=group)

    # 2) prepare indices
    indices = prepare_run_indices(
        split=bundle["split"],
        y=bundle["y"],
        p=p,
        val_ratio=CONFIG["val_ratio"],
        seed=seed,
    )

    # 3) prepare normalized arrays
    arrays = prepare_indexed_arrays(
        bundle=bundle,
        indices_dict=indices,
        fallback_p_for_zero=CONFIG["fallback_p_for_zero"],
    )

    # 4) build loaders
    loaders = build_loaders(
        arrays_dict=arrays,
        batch_size=CONFIG["batch_size"],
        num_workers=CONFIG["num_workers"],
        pin_memory=CONFIG["pin_memory"],
    )

    # 5) build model
    model = build_model(
        scenario=scenario,
        device=device,
        encoder_checkpoint=encoder_checkpoint,
    )

    split_stats = extract_split_stats(indices, bundle["y"])

    # -------------------------------------------------
    # CASE A: p == 0 -> no training
    # -------------------------------------------------
    if p == 0:
        test_result = evaluate_on_test(
            model=model,
            test_loader=loaders["test_loader"],
            device=device,
        )

        result_row = {
            "subject_id": subject_id,
            "group": group,
            "p": p,
            "scenario": scenario,
            "seed": seed,
            "encoder_checkpoint": str(encoder_checkpoint) if encoder_checkpoint is not None else None,

            **split_stats,

            "best_epoch": None,
            "best_val_loss": None,
            "stopped_epoch": None,

            "auc": test_result["metrics"]["auc"],
            "accuracy": test_result["metrics"]["accuracy"],
            "f1": test_result["metrics"]["f1"],
            "precision": test_result["metrics"]["precision"],
            "recall": test_result["metrics"]["recall"],
            "fdr": test_result["metrics"]["fdr"],
        }

        history_df = pd.DataFrame(columns=["epoch", "train_loss", "val_loss"])

        return {
            "result_row": result_row,
            "history_df": history_df,
            "predictions": test_result["predictions"],
        }

    # -------------------------------------------------
    # CASE B: p > 0 -> training + best model + test
    # -------------------------------------------------
    optimizer = build_optimizer(
        model=model,
        lr=CONFIG["lr"],
        weight_decay=CONFIG["weight_decay"],
    )
    criterion = build_criterion()

    fit_result = fit_model(
        model=model,
        train_loader=loaders["train_loader"],
        val_loader=loaders["val_loader"],
        optimizer=optimizer,
        criterion=criterion,
        device=device,
        max_epochs=CONFIG["max_epochs"],
        patience=CONFIG["patience"],
        min_delta=CONFIG["min_delta"],
        verbose=False,
    )

    model = load_best_model_state(model, fit_result)

    test_result = evaluate_on_test(
        model=model,
        test_loader=loaders["test_loader"],
        device=device,
    )

    history_df = pd.DataFrame(fit_result["history"])

    result_row = {
        "subject_id": subject_id,
        "group": group,
        "p": p,
        "scenario": scenario,
        "seed": seed,
        "encoder_checkpoint": str(encoder_checkpoint) if encoder_checkpoint is not None else None,

        **split_stats,

        "best_epoch": fit_result["best_epoch"],
        "best_val_loss": fit_result["best_val_loss"],
        "stopped_epoch": fit_result["stopped_epoch"],

        "auc": test_result["metrics"]["auc"],
        "accuracy": test_result["metrics"]["accuracy"],
        "f1": test_result["metrics"]["f1"],
        "precision": test_result["metrics"]["precision"],
        "recall": test_result["metrics"]["recall"],
        "fdr": test_result["metrics"]["fdr"],
    }

    return {
        "result_row": result_row,
        "history_df": history_df,
        "predictions": test_result["predictions"],
    }

In [49]:
# Для сохранения путей
def make_run_tag(subject_id: str, group: str, p: int, scenario: str, seed: int):
    return f"{group}__{subject_id}__p{p}__{scenario}__seed{seed}"


def ensure_results_dirs(results_root: Path):
    results_root = Path(results_root)
    (results_root / "history").mkdir(parents=True, exist_ok=True)
    (results_root / "predictions").mkdir(parents=True, exist_ok=True)
    (results_root / "tables").mkdir(parents=True, exist_ok=True)
    return results_root

In [50]:
# Сохранение hystory и prediction
def save_history_df(history_df: pd.DataFrame, run_tag: str, results_root: Path):
    results_root = ensure_results_dirs(results_root)
    out_path = results_root / "history" / f"{run_tag}.csv"
    history_df.to_csv(out_path, index=False)
    return out_path


def save_predictions_npz(predictions: dict, run_tag: str, results_root: Path):
    results_root = ensure_results_dirs(results_root)
    out_path = results_root / "predictions" / f"{run_tag}.npz"

    np.savez_compressed(
        out_path,
        y_true=predictions["y_true"],
        prob_score=predictions["prob_score"],
        logit_score=predictions["logit_score"],
        y_pred=predictions["y_pred"],
    )
    return out_path

In [51]:
# Запуск одно run с сохранением
def run_one_and_save(
    subject_id: str,
    p: int,
    scenario: str,
    seed: int,
    group: str,
    results_root: Path,
    encoder_checkpoint: str = None,
    save_history: bool = True,
    save_predictions: bool = True,
):
    """
    Обёртка над run_one(...), которая:
    - запускает эксперимент
    - сохраняет history
    - сохраняет predictions
    - возвращает result_row
    """
    out = run_one(
        subject_id=subject_id,
        p=p,
        scenario=scenario,
        seed=seed,
        group=group,
        encoder_checkpoint=encoder_checkpoint,
    )

    run_tag = make_run_tag(
        subject_id=subject_id,
        group=group,
        p=p,
        scenario=scenario,
        seed=seed,
    )

    history_path = None
    pred_path = None

    if save_history:
        history_path = save_history_df(
            history_df=out["history_df"],
            run_tag=run_tag,
            results_root=results_root,
        )

    if save_predictions:
        pred_path = save_predictions_npz(
            predictions=out["predictions"],
            run_tag=run_tag,
            results_root=results_root,
        )

    result_row = dict(out["result_row"])
    result_row["run_tag"] = run_tag
    result_row["history_path"] = str(history_path) if history_path is not None else None
    result_row["predictions_path"] = str(pred_path) if pred_path is not None else None

    return result_row

In [52]:
def run_many(
    subject_list,
    p_list,
    scenario_list,
    seed_list,
    group: str,
    results_root: Path,
    encoder_checkpoint: str = None,
    save_history: bool = True,
    save_predictions: bool = True,
    save_summary_every: int = 1,
    continue_on_error: bool = True,
):
    """
    Массовый запуск серии экспериментов.

    Возвращает DataFrame с result_row по всем run.
    Промежуточно сохраняет summary_results.csv в /tables.
    """
    results_root = ensure_results_dirs(results_root)
    summary_path = results_root / "tables" / "summary_results.csv"

    rows = []
    run_counter = 0

    total_runs = (
        len(subject_list)
        * len(p_list)
        * len(scenario_list)
        * len(seed_list)
    )

    print(f"Planned runs: {total_runs}")

    for subject_id in subject_list:
        for p in p_list:
            for scenario in scenario_list:
                for seed in seed_list:
                    run_counter += 1
                    print(
                        f"[{run_counter}/{total_runs}] "
                        f"subject={subject_id} | group={group} | p={p} | scenario={scenario} | seed={seed}"
                    )

                    try:
                        row = run_one_and_save(
                            subject_id=subject_id,
                            p=p,
                            scenario=scenario,
                            seed=seed,
                            group=group,
                            results_root=results_root,
                            encoder_checkpoint=encoder_checkpoint if scenario == "ssl_ft" else None,
                            save_history=save_history,
                            save_predictions=save_predictions,
                        )
                        row["status"] = "ok"
                        row["error"] = None

                    except Exception as e:
                        if not continue_on_error:
                            raise

                        row = {
                            "subject_id": subject_id,
                            "group": group,
                            "p": p,
                            "scenario": scenario,
                            "seed": seed,
                            "encoder_checkpoint": str(encoder_checkpoint) if encoder_checkpoint is not None else None,
                            "status": "error",
                            "error": repr(e),
                        }
                        print(f"[ERROR] {row['error']}")

                    rows.append(row)

                    if run_counter % save_summary_every == 0:
                        pd.DataFrame(rows).to_csv(summary_path, index=False)

    results_df = pd.DataFrame(rows)
    results_df.to_csv(summary_path, index=False)
    return results_df

In [53]:
# Функция для стения summary
def load_summary_results(results_root: Path):
    summary_path = Path(results_root) / "tables" / "summary_results.csv"
    if not summary_path.exists():
        raise FileNotFoundError(f"Summary file not found: {summary_path}")
    return pd.read_csv(summary_path)

## БЫСТРЫЙ СТАРТ

In [ ]:
results_df = run_many(
    subject_list=SUBJECT_REGISTRY["train"][:3],
    p_list=CONFIG["p_list"],
    scenario_list=CONFIG["scenario_list"],
    seed_list=[CONFIG["seed"]],
    group="train",
    results_root=CONFIG["results_root"],
    encoder_checkpoint=CONFIG["encoder_checkpoint"],
    save_history=CONFIG["save_history"],
    save_predictions=CONFIG["save_predictions"],
)

## 12. Sanity-check

#### Проверка загрузки данных

In [34]:
# Проверить .npz
def inspect_npz(path: Path):
    data = np.load(path, allow_pickle=True)
    info = {}
    for k in data.files:
        arr = data[k]
        if isinstance(arr, np.ndarray):
            info[k] = {
                "shape": arr.shape,
                "dtype": str(arr.dtype),
            }
        else:
            info[k] = str(type(arr))
    return info

In [35]:
# Проверить .json
def inspect_json(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    return obj

In [36]:
# # example subject
# example_train_subj = SUBJECT_REGISTRY["train"][0]
# print("Example train subject:", example_train_subj)

# # check file existence
# report = check_subject_files(example_train_subj, "train", p_list=(10, 20, 40, 60, 100))
# print(report)

# # inspect epochs npz
# epochs_info = inspect_npz(get_epochs_path(example_train_subj, "train"))
# print("Epochs info:", epochs_info)

# # inspect split json
# split_obj = inspect_json(get_split_path(example_train_subj, "train"))
# print("Split keys:", split_obj.keys() if isinstance(split_obj, dict) else type(split_obj))
# print(split_obj)

# # inspect stats npz
# stats_info = inspect_npz(get_stats_path(example_train_subj, 10, "train"))
# print("Stats info:", stats_info)

In [37]:
bundle = load_subject_bundle(example_train_subj, p=10, group="train")

print("X shape:", bundle["X"].shape)
print("y shape:", bundle["y"].shape)
print("mean shape:", None if bundle["mean"] is None else bundle["mean"].shape)
print("std shape:", None if bundle["std"] is None else bundle["std"].shape)
print("split type:", type(bundle["split"]))

NameError: name 'example_train_subj' is not defined

#### Sanity check for Step 3

In [60]:
# Диагностические функции
def summarize_indices(indices_dict: dict, y: np.ndarray):
    """
    Краткая сводка по размерам и классам в каждом наборе индексов.
    """
    def _stats(idx):
        idx = np.asarray(idx, dtype=np.int64)
        n = len(idx)
        if n == 0:
            return {"n": 0, "pos": 0, "neg": 0}
        n_pos = int(y[idx].sum())
        n_neg = int(n - n_pos)
        return {"n": n, "pos": n_pos, "neg": n_neg}

    return {
        "calib_pool": _stats(indices_dict["calib_pool_idx"]),
        "calib": _stats(indices_dict["calib_idx"]),
        "train": _stats(indices_dict["train_idx"]),
        "val": _stats(indices_dict["val_idx"]),
        "test": _stats(indices_dict["test_idx"]),
    }


def check_no_overlap(indices_dict: dict):
    """
    Проверяет, что train / val / test не пересекаются.
    """
    train_set = set(indices_dict["train_idx"].tolist())
    val_set = set(indices_dict["val_idx"].tolist())
    test_set = set(indices_dict["test_idx"].tolist())

    print("train ∩ val :", len(train_set & val_set))
    print("train ∩ test:", len(train_set & test_set))
    print("val ∩ test  :", len(val_set & test_set))

In [ ]:
# проверка на одном subject и одном p.
subject_id = SUBJECT_REGISTRY["train"][0]
group = "train"
p = 10

bundle = load_subject_bundle(subject_id, p=p, group=group)
X = bundle["X"]
y = bundle["y"]
split = bundle["split"]

indices = prepare_run_indices(
    split=split,
    y=y,
    p=p,
    val_ratio=CONFIG["val_ratio"],
    seed=CONFIG["seed"],
)

print("subject_id:", subject_id)
print("X shape:", X.shape)
print("y shape:", y.shape)
print("summary:", summarize_indices(indices, y))
check_no_overlap(indices)

In [ ]:
# проверка по всем p, включая p=0
subject_id = SUBJECT_REGISTRY["train"][0]
group = "train"

bundle = load_subject_bundle(subject_id, p=10, group=group)
y = bundle["y"]
split = bundle["split"]

for p in [0, 10, 20, 40, 60, 100]:
    indices = prepare_run_indices(
        split=split,
        y=y,
        p=p,
        val_ratio=CONFIG["val_ratio"],
        seed=CONFIG["seed"],
    )
    print(f"\np = {p}")
    print(summarize_indices(indices, y))
    check_no_overlap(indices)

#### Проверка нормализации и нарезки по индексам

In [64]:
def summarize_prepared_arrays(arrays_dict: dict):
    def _info(X, y):
        n = len(y)
        pos = int(y.sum()) if n > 0 else 0
        neg = int(n - pos)
        return {
            "X_shape": X.shape,
            "y_shape": y.shape,
            "n": n,
            "pos": pos,
            "neg": neg,
        }

    return {
        "train": _info(arrays_dict["X_train"], arrays_dict["y_train"]),
        "val": _info(arrays_dict["X_val"], arrays_dict["y_val"]),
        "test": _info(arrays_dict["X_test"], arrays_dict["y_test"]),
        "mean_shape": arrays_dict["mean"].shape,
        "std_shape": arrays_dict["std"].shape,
    }

In [ ]:
# Проверка на одном и suject и одном p
subject_id = SUBJECT_REGISTRY["train"][0]
group = "train"
p = 10

bundle = load_subject_bundle(subject_id, p=p, group=group)

indices = prepare_run_indices(
    split=bundle["split"],
    y=bundle["y"],
    p=p,
    val_ratio=CONFIG["val_ratio"],
    seed=CONFIG["seed"],
)

arrays = prepare_indexed_arrays(
    bundle=bundle,
    indices_dict=indices,
    fallback_p_for_zero=10,
)

print("subject_id:", subject_id)
print("p:", p)
print(summarize_prepared_arrays(arrays))

In [ ]:
# Доп проверка нормализации
print("Train mean (global):", float(arrays["X_train"].mean()) if len(arrays["X_train"]) > 0 else None)
print("Train std  (global):", float(arrays["X_train"].std()) if len(arrays["X_train"]) > 0 else None)

print("Val mean   (global):", float(arrays["X_val"].mean()) if len(arrays["X_val"]) > 0 else None)
print("Val std    (global):", float(arrays["X_val"].std()) if len(arrays["X_val"]) > 0 else None)

print("Test mean  (global):", float(arrays["X_test"].mean()) if len(arrays["X_test"]) > 0 else None)
print("Test std   (global):", float(arrays["X_test"].std()) if len(arrays["X_test"]) > 0 else None)

In [ ]:
# Проверка p=0
subject_id = SUBJECT_REGISTRY["train"][0]
group = "train"
p = 0

bundle = load_subject_bundle(subject_id, p=p, group=group)

indices = prepare_run_indices(
    split=bundle["split"],
    y=bundle["y"],
    p=p,
    val_ratio=CONFIG["val_ratio"],
    seed=CONFIG["seed"],
)

arrays = prepare_indexed_arrays(
    bundle=bundle,
    indices_dict=indices,
    fallback_p_for_zero=10,
)

print("subject_id:", subject_id)
print("p:", p)
print(summarize_prepared_arrays(arrays))

#### Проверка Dataset/Dataloader

In [63]:
def summarize_loaders(loaders_dict: dict):
    def _loader_info(loader):
        if loader is None:
            return {
                "exists": False,
                "n_samples": 0,
                "n_batches": 0,
                "batch_size": None,
            }

        return {
            "exists": True,
            "n_samples": len(loader.dataset),
            "n_batches": len(loader),
            "batch_size": loader.batch_size,
        }

    return {
        "train_loader": _loader_info(loaders_dict["train_loader"]),
        "val_loader": _loader_info(loaders_dict["val_loader"]),
        "test_loader": _loader_info(loaders_dict["test_loader"]),
    }

In [ ]:
subject_id = SUBJECT_REGISTRY["train"][0]
group = "train"
p = 10

bundle = load_subject_bundle(subject_id, p=p, group=group)

indices = prepare_run_indices(
    split=bundle["split"],
    y=bundle["y"],
    p=p,
    val_ratio=CONFIG["val_ratio"],
    seed=CONFIG["seed"],
)

arrays = prepare_indexed_arrays(
    bundle=bundle,
    indices_dict=indices,
    fallback_p_for_zero=10,
)

loaders = build_loaders(
    arrays_dict=arrays,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=True,
)

print("subject_id:", subject_id)
print("p:", p)
print(summarize_loaders(loaders))

In [ ]:
# Проверка одного батча
train_loader = loaders["train_loader"]
xb, yb = next(iter(train_loader))

print("xb shape:", xb.shape)
print("yb shape:", yb.shape)
print("xb dtype:", xb.dtype)
print("yb dtype:", yb.dtype)

In [ ]:
# Проверка для р=0
subject_id = SUBJECT_REGISTRY["train"][0]
group = "train"
p = 0

bundle = load_subject_bundle(subject_id, p=p, group=group)

indices = prepare_run_indices(
    split=bundle["split"],
    y=bundle["y"],
    p=p,
    val_ratio=CONFIG["val_ratio"],
    seed=CONFIG["seed"],
)

arrays = prepare_indexed_arrays(
    bundle=bundle,
    indices_dict=indices,
    fallback_p_for_zero=10,
)

loaders = build_loaders(
    arrays_dict=arrays,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
)

print("subject_id:", subject_id)
print("p:", p)
print(summarize_loaders(loaders))

#### Проверка модели

In [ ]:
device = CONFIG["device"]

model_scratch = build_model(
    scenario="scratch",
    device=device,
    encoder_checkpoint=None,
)

print("All params:", count_all_parameters(model_scratch))
print("Trainable params:", count_trainable_parameters(model_scratch))

In [ ]:
subject_id = SUBJECT_REGISTRY["train"][0]
group = "train"
p = 10

bundle = load_subject_bundle(subject_id, p=p, group=group)

indices = prepare_run_indices(
    split=bundle["split"],
    y=bundle["y"],
    p=p,
    val_ratio=CONFIG["val_ratio"],
    seed=CONFIG["seed"],
)

arrays = prepare_indexed_arrays(
    bundle=bundle,
    indices_dict=indices,
    fallback_p_for_zero=10,
)

loaders = build_loaders(
    arrays_dict=arrays,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
)

xb, yb = next(iter(loaders["train_loader"]))
xb = xb.to(device)

with torch.no_grad():
    logits = model_scratch(xb)

print("xb shape:", xb.shape)
print("logits shape:", logits.shape)
print("logits dtype:", logits.dtype)

In [ ]:
model_ssl = build_model(
    scenario="ssl_ft",
    device=device,
    encoder_checkpoint=CONFIG["encoder_checkpoint"],
)

with torch.no_grad():
    logits_ssl = model_ssl(xb)

print("SSL model built successfully.")
print("logits_ssl shape:", logits_ssl.shape)
print("logits_ssl dtype:", logits_ssl.dtype)

#### Тестовый прогон на одном испытуемом

In [ ]:
device = CONFIG["device"]

subject_id = SUBJECT_REGISTRY["train"][0]
group = "train"
p = 10
scenario = "scratch"

bundle = load_subject_bundle(subject_id, p=p, group=group)

indices = prepare_run_indices(
    split=bundle["split"],
    y=bundle["y"],
    p=p,
    val_ratio=CONFIG["val_ratio"],
    seed=CONFIG["seed"],
)

arrays = prepare_indexed_arrays(
    bundle=bundle,
    indices_dict=indices,
    fallback_p_for_zero=10,
)

loaders = build_loaders(
    arrays_dict=arrays,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
)

model = build_model(
    scenario=scenario,
    device=device,
    encoder_checkpoint=None,
)

optimizer = build_optimizer(
    model=model,
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)

criterion = build_criterion()

history = fit_model(
    model=model,
    train_loader=loaders["train_loader"],
    val_loader=loaders["val_loader"],
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    max_epochs=3,
    verbose=True,
)

In [ ]:
history_df = pd.DataFrame(history)
history_df

In [ ]:
# быстрый sanity-check для ssl_ft
device = CONFIG["device"]

subject_id = SUBJECT_REGISTRY["train"][0]
group = "train"
p = 10
scenario = "ssl_ft"

bundle = load_subject_bundle(subject_id, p=p, group=group)

indices = prepare_run_indices(
    split=bundle["split"],
    y=bundle["y"],
    p=p,
    val_ratio=CONFIG["val_ratio"],
    seed=CONFIG["seed"],
)

arrays = prepare_indexed_arrays(
    bundle=bundle,
    indices_dict=indices,
    fallback_p_for_zero=10,
)

loaders = build_loaders(
    arrays_dict=arrays,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
)

model = build_model(
    scenario=scenario,
    device=device,
    encoder_checkpoint=CONFIG["encoder_checkpoint"],
)

optimizer = build_optimizer(
    model=model,
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)

criterion = build_criterion()

history_ssl = fit_model(
    model=model,
    train_loader=loaders["train_loader"],
    val_loader=loaders["val_loader"],
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    max_epochs=3,
    verbose=True,
)

#### Короткий прогон уже с early stopping.

In [ ]:
device = CONFIG["device"]

subject_id = SUBJECT_REGISTRY["train"][0]
group = "train"
p = 10
scenario = "scratch"

bundle = load_subject_bundle(subject_id, p=p, group=group)

indices = prepare_run_indices(
    split=bundle["split"],
    y=bundle["y"],
    p=p,
    val_ratio=CONFIG["val_ratio"],
    seed=CONFIG["seed"],
)

arrays = prepare_indexed_arrays(
    bundle=bundle,
    indices_dict=indices,
    fallback_p_for_zero=CONFIG["fallback_p_for_zero"],
)

loaders = build_loaders(
    arrays_dict=arrays,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
)

model = build_model(
    scenario=scenario,
    device=device,
    encoder_checkpoint=None,
)

optimizer = build_optimizer(
    model=model,
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)

criterion = build_criterion()

fit_result = fit_model(
    model=model,
    train_loader=loaders["train_loader"],
    val_loader=loaders["val_loader"],
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    max_epochs=CONFIG["max_epochs"],
    patience=CONFIG["patience"],
    min_delta=CONFIG["min_delta"],
    verbose=True,
)

print("best_epoch:", fit_result["best_epoch"])
print("best_val_loss:", fit_result["best_val_loss"])
print("stopped_epoch:", fit_result["stopped_epoch"])

In [ ]:
history_df = pd.DataFrame(fit_result["history"])
history_df.tail()

In [ ]:
model = load_best_model_state(model, fit_result)
print("Best model state loaded.")

In [ ]:
# Для ssl_ft
device = CONFIG["device"]

subject_id = SUBJECT_REGISTRY["train"][0]
group = "train"
p = 10
scenario = "ssl_ft"

bundle = load_subject_bundle(subject_id, p=p, group=group)

indices = prepare_run_indices(
    split=bundle["split"],
    y=bundle["y"],
    p=p,
    val_ratio=CONFIG["val_ratio"],
    seed=CONFIG["seed"],
)

arrays = prepare_indexed_arrays(
    bundle=bundle,
    indices_dict=indices,
    fallback_p_for_zero=CONFIG["fallback_p_for_zero"],
)

loaders = build_loaders(
    arrays_dict=arrays,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
)

model_ssl = build_model(
    scenario=scenario,
    device=device,
    encoder_checkpoint=CONFIG["encoder_checkpoint"],
)

optimizer_ssl = build_optimizer(
    model=model_ssl,
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)

criterion = build_criterion()

fit_result_ssl = fit_model(
    model=model_ssl,
    train_loader=loaders["train_loader"],
    val_loader=loaders["val_loader"],
    optimizer=optimizer_ssl,
    criterion=criterion,
    device=device,
    max_epochs=CONFIG["max_epochs"],
    patience=CONFIG["patience"],
    min_delta=CONFIG["min_delta"],
    verbose=True,
)

print("best_epoch:", fit_result_ssl["best_epoch"])
print("best_val_loss:", fit_result_ssl["best_val_loss"])
print("stopped_epoch:", fit_result_ssl["stopped_epoch"])

#### Проверка подсчёта метрик

In [46]:
device = CONFIG["device"]

subject_id = SUBJECT_REGISTRY["train"][0]
group = "train"
p = 10
scenario = "scratch"

bundle = load_subject_bundle(subject_id, p=p, group=group)

indices = prepare_run_indices(
    split=bundle["split"],
    y=bundle["y"],
    p=p,
    val_ratio=CONFIG["val_ratio"],
    seed=CONFIG["seed"],
)

arrays = prepare_indexed_arrays(
    bundle=bundle,
    indices_dict=indices,
    fallback_p_for_zero=CONFIG["fallback_p_for_zero"],
)

loaders = build_loaders(
    arrays_dict=arrays,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
)

model = build_model(
    scenario=scenario,
    device=device,
    encoder_checkpoint=None,
)

optimizer = build_optimizer(
    model=model,
    lr=CONFIG["lr"],
    weight_decay=CONFIG["weight_decay"],
)

criterion = build_criterion()

fit_result = fit_model(
    model=model,
    train_loader=loaders["train_loader"],
    val_loader=loaders["val_loader"],
    optimizer=optimizer,
    criterion=criterion,
    device=device,
    max_epochs=CONFIG["max_epochs"],
    patience=CONFIG["patience"],
    min_delta=CONFIG["min_delta"],
    verbose=False,
)

model = load_best_model_state(model, fit_result)

test_result = evaluate_on_test(
    model=model,
    test_loader=loaders["test_loader"],
    device=device,
)

print("best_epoch:", fit_result["best_epoch"])
print("best_val_loss:", fit_result["best_val_loss"])
print("metrics:", test_result["metrics"])

best_epoch: 3
best_val_loss: 0.3311748579556112
metrics: {'auc': 0.538515006478964, 'accuracy': 0.9014702089244261, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'fdr': 0.004080749318428614}


In [48]:
pred = test_result["predictions"]

print("y_true shape     :", pred["y_true"].shape)
print("prob_score shape :", pred["prob_score"].shape)
print("logit_score shape:", pred["logit_score"].shape)
print("y_pred shape     :", pred["y_pred"].shape)

print("prob_score min/max :", float(pred["prob_score"].min()), float(pred["prob_score"].max()))
print("logit_score min/max:", float(pred["logit_score"].min()), float(pred["logit_score"].max()))
print("positive preds     :", int(pred["y_pred"].sum()))

y_true shape     : (7754,)
prob_score shape : (7754,)
logit_score shape: (7754,)
y_pred shape     : (7754,)
prob_score min/max : 5.2569492254406214e-05 0.2973318099975586
logit_score min/max: -4.823914051055908 -0.3995714485645294
positive preds     : 0


#### Проверка run_one

In [51]:
# тест scratch, p=10
run_out = run_one(
    subject_id=SUBJECT_REGISTRY["train"][0],
    p=10,
    scenario="scratch",
    seed=CONFIG["seed"],
    group="train",
    encoder_checkpoint=None,
)

print(run_out["result_row"])
print(run_out["history_df"].head())
print(run_out["history_df"].tail())

{'subject_id': 'subj_000', 'group': 'train', 'p': 10, 'scenario': 'scratch', 'seed': 42, 'encoder_checkpoint': None, 'n_calib': 1809, 'n_val': 362, 'n_test': 7754, 'n_pos_calib': 168, 'n_pos_val': 34, 'n_pos_test': 764, 'best_epoch': 3, 'best_val_loss': 0.31649407974922855, 'stopped_epoch': 13, 'auc': 0.5645116621351369, 'accuracy': 0.9008253804488007, 'f1': 0.00517464424320828, 'precision': 0.2222222222222222, 'recall': 0.002617801047120419, 'fdr': 0.011909714609070823}
   epoch  train_loss  val_loss
0      1    0.518574  0.365050
1      2    0.304163  0.324321
2      3    0.276534  0.316494
3      4    0.223910  0.394616
4      5    0.174095  0.439386
    epoch  train_loss  val_loss
8       9    0.047193  0.425960
9      10    0.035737  0.451140
10     11    0.028058  0.428825
11     12    0.022333  0.534145
12     13    0.018804  0.471666


In [53]:
# shapes predictions
pred = run_out["predictions"]

print("y_true shape     :", pred["y_true"].shape)
print("prob_score shape :", pred["prob_score"].shape)
print("logit_score shape:", pred["logit_score"].shape)
print("y_pred shape     :", pred["y_pred"].shape)

y_true shape     : (7754,)
prob_score shape : (7754,)
logit_score shape: (7754,)
y_pred shape     : (7754,)


In [54]:
# тест р=0
run_out_p0 = run_one(
    subject_id=SUBJECT_REGISTRY["train"][0],
    p=0,
    scenario="scratch",
    seed=CONFIG["seed"],
    group="train",
    encoder_checkpoint=None,
)

print(run_out_p0["result_row"])
print(run_out_p0["history_df"])

{'subject_id': 'subj_000', 'group': 'train', 'p': 0, 'scenario': 'scratch', 'seed': 42, 'encoder_checkpoint': None, 'n_calib': 0, 'n_val': 0, 'n_test': 7754, 'n_pos_calib': 0, 'n_pos_val': 0, 'n_pos_test': 764, 'best_epoch': None, 'best_val_loss': None, 'stopped_epoch': None, 'auc': 0.44429944797728993, 'accuracy': 0.9014702089244261, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'fdr': 0.0011749428286297818}
Empty DataFrame
Columns: [epoch, train_loss, val_loss]
Index: []


In [55]:
# тест ssl_ft
run_out_ssl = run_one(
    subject_id=SUBJECT_REGISTRY["train"][0],
    p=10,
    scenario="ssl_ft",
    seed=CONFIG["seed"],
    group="train",
    encoder_checkpoint=CONFIG["encoder_checkpoint"],
)

print(run_out_ssl["result_row"])
print(run_out_ssl["history_df"].tail())

{'subject_id': 'subj_000', 'group': 'train', 'p': 10, 'scenario': 'ssl_ft', 'seed': 42, 'encoder_checkpoint': '/kaggle/input/datasets/taisiyaglazova/ssl-full-encoder-best/encoder_best.pt', 'n_calib': 1809, 'n_val': 362, 'n_test': 7754, 'n_pos_calib': 168, 'n_pos_val': 34, 'n_pos_test': 764, 'best_epoch': 6, 'best_val_loss': 0.326184702512309, 'stopped_epoch': 16, 'auc': 0.5555254701930207, 'accuracy': 0.9014702089244261, 'f1': 0.0, 'precision': 0.0, 'recall': 0.0, 'fdr': 0.019551264151360114}
    epoch  train_loss  val_loss
11     12    0.166099  0.363693
12     13    0.147392  0.384066
13     14    0.131356  0.391144
14     15    0.111465  0.396032
15     16    0.098354  0.409998


#### Проверка run_many

In [54]:
small_subject_list = [SUBJECT_REGISTRY["train"][0]]
small_p_list = [0, 10]
small_scenarios = ["scratch", "ssl_ft"]
small_seeds = [CONFIG["seed"]]

results_df = run_many(
    subject_list=small_subject_list,
    p_list=small_p_list,
    scenario_list=small_scenarios,
    seed_list=small_seeds,
    group="train",
    results_root=CONFIG["results_root"],
    encoder_checkpoint=CONFIG["encoder_checkpoint"],
    save_history=True,
    save_predictions=True,
    save_summary_every=1,
    continue_on_error=True,
)

results_df

Planned runs: 4
[1/4] subject=subj_000 | group=train | p=0 | scenario=scratch | seed=42
[2/4] subject=subj_000 | group=train | p=0 | scenario=ssl_ft | seed=42
[3/4] subject=subj_000 | group=train | p=10 | scenario=scratch | seed=42
[4/4] subject=subj_000 | group=train | p=10 | scenario=ssl_ft | seed=42


,subject_id,group,p,scenario,seed,encoder_checkpoint,n_calib,n_val,n_test,n_pos_calib,...,accuracy,f1,precision,recall,fdr,run_tag,history_path,predictions_path,status,error
0,subj_000,train,0,scratch,42,None,0,0,7754,0,...,0.901470,0.000000,0.000000,0.000000,0.001175,train__subj_000__p0__scratch__seed42,/kaggle/working/stage5_block2_results/history/...,/kaggle/working/stage5_block2_results/predicti...,ok,None
1,subj_000,train,0,ssl_ft,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,0,0,7754,0,...,0.098530,0.179385,0.098530,1.000000,0.002301,train__subj_000__p0__ssl_ft__seed42,/kaggle/working/stage5_block2_results/history/...,/kaggle/working/stage5_block2_results/predicti...,ok,None
2,subj_000,train,10,scratch,42,None,1809,362,7754,168,...,0.900825,0.005175,0.222222,0.002618,0.011910,train__subj_000__p10__scratch__seed42,/kaggle/working/stage5_block2_results/history/...,/kaggle/working/stage5_block2_results/predicti...,ok,None
3,subj_000,train,10,ssl_ft,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,1809,362,7754,168,...,0.901470,0.000000,0.000000,0.000000,0.019551,train__subj_000__p10__ssl_ft__seed42,/kaggle/working/stage5_block2_results/history/...,/kaggle/working/stage5_block2_results/predicti...,ok,None


In [55]:
# Посмотрим, что сохранилось
print("History files:")
print(sorted((CONFIG["results_root"] / "history").glob("*"))[:10])

print("\nPrediction files:")
print(sorted((CONFIG["results_root"] / "predictions").glob("*"))[:10])

print("\nSummary table:")
summary_df = load_summary_results(CONFIG["results_root"])
summary_df

History files:
[PosixPath('/kaggle/working/stage5_block2_results/history/train__subj_000__p0__scratch__seed42.csv'), PosixPath('/kaggle/working/stage5_block2_results/history/train__subj_000__p0__ssl_ft__seed42.csv'), PosixPath('/kaggle/working/stage5_block2_results/history/train__subj_000__p10__scratch__seed42.csv'), PosixPath('/kaggle/working/stage5_block2_results/history/train__subj_000__p10__ssl_ft__seed42.csv')]

Prediction files:
[PosixPath('/kaggle/working/stage5_block2_results/predictions/train__subj_000__p0__scratch__seed42.npz'), PosixPath('/kaggle/working/stage5_block2_results/predictions/train__subj_000__p0__ssl_ft__seed42.npz'), PosixPath('/kaggle/working/stage5_block2_results/predictions/train__subj_000__p10__scratch__seed42.npz'), PosixPath('/kaggle/working/stage5_block2_results/predictions/train__subj_000__p10__ssl_ft__seed42.npz')]

Summary table:


,subject_id,group,p,scenario,seed,encoder_checkpoint,n_calib,n_val,n_test,n_pos_calib,...,accuracy,f1,precision,recall,fdr,run_tag,history_path,predictions_path,status,error
0,subj_000,train,0,scratch,42,NaN,0,0,7754,0,...,0.901470,0.000000,0.000000,0.000000,0.001175,train__subj_000__p0__scratch__seed42,/kaggle/working/stage5_block2_results/history/...,/kaggle/working/stage5_block2_results/predicti...,ok,NaN
1,subj_000,train,0,ssl_ft,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,0,0,7754,0,...,0.098530,0.179385,0.098530,1.000000,0.002301,train__subj_000__p0__ssl_ft__seed42,/kaggle/working/stage5_block2_results/history/...,/kaggle/working/stage5_block2_results/predicti...,ok,NaN
2,subj_000,train,10,scratch,42,NaN,1809,362,7754,168,...,0.900825,0.005175,0.222222,0.002618,0.011910,train__subj_000__p10__scratch__seed42,/kaggle/working/stage5_block2_results/history/...,/kaggle/working/stage5_block2_results/predicti...,ok,NaN
3,subj_000,train,10,ssl_ft,42,/kaggle/input/datasets/taisiyaglazova/ssl-full...,1809,362,7754,168,...,0.901470,0.000000,0.000000,0.000000,0.019551,train__subj_000__p10__ssl_ft__seed42,/kaggle/working/stage5_block2_results/history/...,/kaggle/working/stage5_block2_results/predicti...,ok,NaN


#### sanity-check на benchmark group

In [56]:
print("n benchmark subjects:", len(SUBJECT_REGISTRY["benchmark"]))
print("example benchmark subjects:", SUBJECT_REGISTRY["benchmark"][:5])

n benchmark subjects: 65
example benchmark subjects: ['subj_051', 'subj_052', 'subj_053', 'subj_054', 'subj_055']


In [57]:
benchmark_subj = SUBJECT_REGISTRY["benchmark"][0]

bundle_bm = load_subject_bundle(
    subject_id=benchmark_subj,
    p=10,
    group="benchmark",
)

print("subject_id:", benchmark_subj)
print("X shape:", bundle_bm["X"].shape)
print("y shape:", bundle_bm["y"].shape)
print("mean shape:", bundle_bm["mean"].shape if bundle_bm["mean"] is not None else None)
print("std shape:", bundle_bm["std"].shape if bundle_bm["std"] is not None else None)
print("split type:", type(bundle_bm["split"]))
print("split keys:", bundle_bm["split"].keys())

subject_id: subj_051
X shape: (8900, 14, 208)
y shape: (8900,)
mean shape: (14,)
std shape: (14,)
split type: <class 'dict'>
split keys: dict_keys(['calib_pool_idx', 'test_rest_idx', 'calib_idx', 'seed', 'sizes', 'class_counts'])


In [61]:
indices_bm = prepare_run_indices(
    split=bundle_bm["split"],
    y=bundle_bm["y"],
    p=10,
    val_ratio=CONFIG["val_ratio"],
    seed=CONFIG["seed"],
)

print(summarize_indices(indices_bm, bundle_bm["y"]))
check_no_overlap(indices_bm)

{'calib_pool': {'n': 6230, 'pos': 740, 'neg': 5490}, 'calib': {'n': 623, 'pos': 74, 'neg': 549}, 'train': {'n': 498, 'pos': 59, 'neg': 439}, 'val': {'n': 125, 'pos': 15, 'neg': 110}, 'test': {'n': 2670, 'pos': 314, 'neg': 2356}}
train ∩ val : 0
train ∩ test: 0
val ∩ test  : 0


In [65]:
arrays_bm = prepare_indexed_arrays(
    bundle=bundle_bm,
    indices_dict=indices_bm,
    fallback_p_for_zero=CONFIG["fallback_p_for_zero"],
)

print(summarize_prepared_arrays(arrays_bm))

loaders_bm = build_loaders(
    arrays_dict=arrays_bm,
    batch_size=CONFIG["batch_size"],
    num_workers=CONFIG["num_workers"],
    pin_memory=CONFIG["pin_memory"],
)

print(summarize_loaders(loaders_bm))

{'train': {'X_shape': (498, 14, 208), 'y_shape': (498,), 'n': 498, 'pos': 59, 'neg': 439}, 'val': {'X_shape': (125, 14, 208), 'y_shape': (125,), 'n': 125, 'pos': 15, 'neg': 110}, 'test': {'X_shape': (2670, 14, 208), 'y_shape': (2670,), 'n': 2670, 'pos': 314, 'neg': 2356}, 'mean_shape': (14,), 'std_shape': (14,)}
{'train_loader': {'exists': True, 'n_samples': 498, 'n_batches': 8, 'batch_size': 64}, 'val_loader': {'exists': True, 'n_samples': 125, 'n_batches': 2, 'batch_size': 64}, 'test_loader': {'exists': True, 'n_samples': 2670, 'n_batches': 42, 'batch_size': 64}}


In [66]:
device = CONFIG["device"]

model_bm = build_model(
    scenario="scratch",
    device=device,
    encoder_checkpoint=None,
)

xb_bm, yb_bm = next(iter(loaders_bm["train_loader"]))
xb_bm = xb_bm.to(device)

with torch.no_grad():
    logits_bm = model_bm(xb_bm)

print("xb shape:", xb_bm.shape)
print("logits shape:", logits_bm.shape)
print("logits dtype:", logits_bm.dtype)

xb shape: torch.Size([64, 14, 208])
logits shape: torch.Size([64, 2])
logits dtype: torch.float32


In [67]:
run_out_bm = run_one(
    subject_id=benchmark_subj,
    p=10,
    scenario="scratch",
    seed=CONFIG["seed"],
    group="benchmark",
    encoder_checkpoint=None,
)

print(run_out_bm["result_row"])
print(run_out_bm["history_df"].head())
print(run_out_bm["history_df"].tail())

{'subject_id': 'subj_051', 'group': 'benchmark', 'p': 10, 'scenario': 'scratch', 'seed': 42, 'encoder_checkpoint': None, 'n_calib': 623, 'n_val': 125, 'n_test': 2670, 'n_pos_calib': 74, 'n_pos_val': 15, 'n_pos_test': 314, 'best_epoch': 6, 'best_val_loss': 0.377597407579422, 'stopped_epoch': 16, 'auc': 0.550909454651628, 'accuracy': 0.8790262172284644, 'f1': 0.0182370820668693, 'precision': 0.2, 'recall': 0.009554140127388535, 'fdr': 0.018976740812656524}
   epoch  train_loss  val_loss
0      1    0.767612  0.743322
1      2    0.537832  0.692734
2      3    0.406665  0.537895
3      4    0.304324  0.450780
4      5    0.253658  0.421825
    epoch  train_loss  val_loss
11     12    0.058969  0.520675
12     13    0.047216  0.464890
13     14    0.043003  0.520698
14     15    0.033454  0.489781
15     16    0.032819  0.490159
